In [ ]:
import os
DATA_ROOT = os.environ.get('WRH_DATA_ROOT', 'G:/WRH_data')  # local data folder; see README
import cdsapi
from pathlib import Path

SAVE_DIR = Path(rf"{DATA_ROOT}/AIS_processing/weather_data/wave")   # same directory as existing 2002-2021
SAVE_DIR.mkdir(parents=True, exist_ok=True)

dataset = "reanalysis-era5-single-levels"
client  = cdsapi.Client()

MONTHS = [f"{m:02d}" for m in range(1, 13)]
DAYS   = [f"{d:02d}" for d in range(1, 32)]
HOURS  = [f"{h:02d}:00" for h in range(24)]

for year in range(1985, 2002):                      # ← fill in only these 17 years
    target = SAVE_DIR / f"era5_swh_{year}.nc"       # ← naming matches existing files
    if target.exists():
        print(f"[SKIP]  {year}"); continue

    print(f"[QUEUE] {year} ...", flush=True)
    request = {
        "product_type": ["reanalysis"],
        "variable": ["significant_height_of_combined_wind_waves_and_swell"],
        "year":  [str(year)],
        "month": MONTHS,
        "day":   DAYS,
        "time":  HOURS,
        "data_format":     "netcdf",
        "download_format": "unarchived",
    }
    try:
        client.retrieve(dataset, request).download(target=str(target))
        print(f"[DONE]  {year}  ({target.stat().st_size/1e9:.1f} GB)", flush=True)
    except Exception as e:
        if target.exists(): target.unlink()          # delete partial file so a rerun can resume
        print(f"[FAIL]  {year}  {type(e).__name__}: {e}", flush=True)

print("\nDone. Directory summary:")
fs = sorted(SAVE_DIR.glob("era5_swh_*.nc"))
print(f"  {len(fs)} files  {sum(f.stat().st_size for f in fs)/1e9:.1f} GB  "
      f"{fs[0].stem[-4:]} - {fs[-1].stem[-4:]}")


In [ ]:
import xarray as xr
from pathlib import Path
for f in sorted(Path(rf"{DATA_ROOT}/AIS_processing/weather_data/wave").glob("era5_swh_*.nc")):
    y = int(f.stem[-4:])
    d = xr.open_dataset(f)
    n = d.sizes['valid_time']; exp = 8784 if y%4==0 and (y%100!=0 or y%400==0) else 8760
    print(f"{y}  {n:5d}/{exp}  {'OK' if n==exp else '★MISSING'}  {d.sizes['latitude']}x{d.sizes['longitude']}")
    d.close()
